# Dorna Pipette — Bench Test Notebook

Interactive test of the Dorna pipette over serial, using the
`dorna_pipette` package in this repo.

**Setup:** from the repo root run `pip install -e .` once, then plug the pipette's
RS-232/USB adapter in and run the cells top to bottom.

**Protocol summary** (baud 38400, 8N1):

| Action | Command | Reply |
|---|---|---|
| Query status | `{addr}>?` | `{addr}<0` idle, `{addr}<1` busy/moving |
| Initialize (home only) | `{addr}>It{speed},100,1` | `{addr}<2` = accepted |
| Home + eject tip | `{addr}>It{speed},100,0` | `{addr}<2` = accepted |
| Aspirate | `{addr}>Ia{steps},{speed},10` | `{addr}<2` = accepted |
| Dispense | `{addr}>Da{steps},0,{speed},{stop_speed}` | `{addr}<2` = accepted; high stop speed = blowout |
| Tip presence | `{addr}>Rr3` | `{addr}<2:1` = tip on, `{addr}<2:0` = no tip |

Error replies: `{addr}<10` = parameter out of range, `{addr}<14` = invalid register.

**Run order:** config → connect → initialize → tip check → aspirate → dispense → eject → close.

## 1. Config

In [ ]:
from dorna_pipette import DornaPipette

# Serial port, e.g. '/dev/ttyUSB0' on Linux or 'COM3' on Windows.
# The next cell lists what's plugged in. On Linux, prefer a stable
# /dev/serial/by-id/... path — USB adapters can re-enumerate
# (ttyUSB0 -> ttyUSB1) between sessions.
PORT = '/dev/ttyUSB0'

# Device address. Fresh units are usually 1.
ADDR = 1

# CALIBRATION — steps per microliter. Depends on the barrel size:
#   1 mL barrel: 5000-step full stroke → 5 steps/µL (1 step = 0.2 µL)
# Over-range aspirate commands are rejected with '{addr}<10' and no motion.
# Confirm with the gravimetric calibration cell before precision work.
FULL_STROKE_STEPS = 5000
BARREL_UL = 1000
STEPS_PER_UL = FULL_STROKE_STEPS / BARREL_UL   # = 5

## 2. List serial ports
See what's plugged in, then set `PORT` in the config cell above.

In [ ]:
import serial.tools.list_ports

ports = serial.tools.list_ports.comports()
if not ports:
    print('No serial ports found — check USB cable / adapter.')
for p in ports:
    print(f'{p.device:20s}  {p.description}  [{p.hwid}]')

## 3. Connect

In [ ]:
pip = DornaPipette(PORT, addr=ADDR, steps_per_ul=STEPS_PER_UL)
print('Connected.' if pip.connect() else
      f'Connect failed — check PORT ({PORT}) and ADDR ({ADDR}) in the config cell.')

## 4. Status & tip check

In [ ]:
print('Status:', pip.status())
tip = pip.has_tip()
print({True: 'Tip is ON', False: 'No tip', None: 'No valid response (device unavailable?)'}[tip])

## 5. Initialize (home the plunger)
Do this once after power-up, before any aspirate/dispense. Keeps the tip if one is on.

In [ ]:
print('Initialize OK:', pip.initialize())

## 6. Aspirate
Press a tip on first (manually for bench testing). Start small.

In [ ]:
VOLUME_UL = 20   # start small
print('Aspirate OK:', pip.aspirate(VOLUME_UL, speed=200))

## 7. Dispense
Set `blowout=True` on the final dispense to push out residual liquid at high stop speed.

In [ ]:
print('Dispense OK:', pip.dispense(VOLUME_UL, speed=500, blowout=True))

## 8. Eject tip

In [ ]:
print('Eject OK:', pip.eject_tip())
print('Tip after eject:', pip.has_tip())

## 9. Gravimetric calibration check

Confirms `STEPS_PER_UL` with a scale. Water is ~1.000 mg/µL at room temp, so
dispensed mass in mg ≈ volume in µL.

1. Put a tip on, put a tared container of water on a scale.
2. Run the cell — it aspirates a known **step count** (not µL) and dispenses into a tared vessel.
3. Enter the measured mass; the cell prints the actual steps/µL.

If the measured value is far from `STEPS_PER_UL`, update the config cell and re-run.

In [ ]:
# Aspirate/dispense by RAW STEPS so the result is independent of STEPS_PER_UL.
CAL_STEPS = FULL_STROKE_STEPS // 2   # half stroke — keeps clear of both ends of travel

print(f'Aspirating {CAL_STEPS} steps of liquid...')
ok = (pip.send(f'Ia{CAL_STEPS},200,10', verbose=True).startswith(f'{ADDR}<2')
      and pip._wait_until_idle())
print('aspirate OK:', ok)

input('Move tip over the tared weigh vessel, then press Enter...')
ok = (pip.send(f'Da{CAL_STEPS},0,500,500', verbose=True).startswith(f'{ADDR}<2')
      and pip._wait_until_idle())
print('dispense OK:', ok)

mass_mg = float(input('Measured dispensed mass (mg): '))
measured_steps_per_ul = CAL_STEPS / mass_mg
print(f'\nActual: {measured_steps_per_ul:.2f} steps/µL '
      f'(config says {STEPS_PER_UL:.2f})')
print(f'Implied barrel size: {FULL_STROKE_STEPS / measured_steps_per_ul:.0f} µL')

## 10. Full cycle test
One complete cycle: aspirate → dispense. Repeat `N_CYCLES` times for a soak test.

In [ ]:
N_CYCLES = 3
VOL = 50

for i in range(N_CYCLES):
    print(f'--- cycle {i + 1}/{N_CYCLES} ---')
    ok = (pip.aspirate(VOL)
          and pip.dispense(VOL, blowout=True))
    print('cycle OK:', ok)
    if not ok:
        print('Stopping — check device.')
        break

## 11. Raw command console
For poking at the protocol directly — the address prefix is added for you.
E.g. `?` (status), `Rr3` (tip register), `It16000,100,1` (home).

In [ ]:
pip.send('?', verbose=True)

## 12. Close

In [ ]:
pip.close()
print('Closed.')